In [1]:
import pandas as pd
import numpy as np
import sys
import glob

In [2]:
sys.path.append("/Users/pnr5sh/Documents/phd/mmmp/")
import sidchaini.sidhelpers as sidhelpers

In [7]:
#reading in meta data from my dir
dataset = pd.read_csv('multipeak_dataset_metadata.csv', header='infer')
dataset.columns

Index(['wise_objid', 'IAU name', 'Internal name/s', 'Obj. RA', 'Obj. DEC',
       'Obj. Type', 'Redshift', 'Spec. ID', 'Obs-date', 'JD', 'Phase (days)',
       'From', 'Telescope', 'Instrument', 'Observer/s', 'Reducer/s',
       'Source group', 'Public', 'Associated groups', 'End prop. period',
       'Ascii file', 'Fits file', 'Spec. type', 'Spec. quality',
       'Extinction-Corrected', 'WL Medium', 'WL Units',
       'Flux Unit Coefficient', 'Spec. units', 'Flux Calibrated By',
       'Exp-time', 'Aperture (slit)', 'HA', 'Airmass', 'Dichroic', 'Grism',
       'Grating', 'Blaze', 'Lambda-min', 'Lambda-max', 'Del-Lambda', 'Contrib',
       'Publish', 'Remarks', 'Created by', 'Creation date', 'mjd', 'peak_mjd',
       'peak_mag', 'peak_filt', 'double-peaked'],
      dtype='object')

In [126]:
dataset.loc[dataset['IAU name']=='SN 2024zsw', 'Internal name/s'] = 'ZTF24abpdzvm'
dataset.loc[dataset['IAU name']=='SN 2020urc', 'Internal name/s'] = 'ZTF20acgiglu'

In [127]:
#maven needs light curves files w/ 4 columns: time,mag,magerr,band
#   where time is in mjd and filter is either g, R

In [ ]:
# dropping bad obs
# centering LCs around date of first spectra (-50, +150 days)
# creating mag and mag_err cols
# dropping points w/ e_mag>2 mags
# creating mjd col

def create_ztf_lc_dfs(datadir, sn_type, check_names=False, save_df=True):
        files = sorted(glob.glob(datadir+'*_fp_lc.txt'))
        ztf_info_df = pd.read_csv(f'ztf_info_files/ztf_fp_info_for_{sn_type}.csv')

        if (sn_type == 'SN IIb') or (sn_type =='SN IIn') or (sn_type == 'SN Ibc'):
                startindx = 18
        elif (sn_type == 'SN Ic') or (sn_type =='SN Ib'):
                startindx = 17
        elif (sn_type == 'SLSN-I') or (sn_type == 'SN IbnIcn') or (sn_type == 'SN Ic-pec'):
                startindx = 20
        elif (sn_type == 'SLSN-II') or (sn_type =='SN Ibc Ca-rich'):
                startindx = 21
        elif sn_type == 'FBOT':
                startindx = 19
        else:
                print(f'SN type {sn_type} unsupported or not in format SN XX / SLSN-XX')
                return

        sn_names, lc_dfs = [],[]
        for file in files:
                sn_name = file[startindx:-10] #NOTE: FIRST INDEX CHANGES W/ DATADIR NAME LENGTH
                if check_names:
                        print(sn_name)
                        continue
                
                sn_names.append(sn_name)

                cols = ['index', 'field', 'ccdid', 'qid', 'filter', 'pid', 'infobitssci', 'sciinpseeing', 'scibckgnd', 'scisigpix',
                        'zpmaginpsci', 'zpmaginpsciunc', 'zpmaginpscirms', 'clrcoeff', 'clrcoeffunc', 'ncalmatches', 'exptime',
                        'adpctdif1', 'adpctdif2', 'diffmaglim', 'zpdiff', 'programid', 'jd', 'rfid', 'forcediffimflux', 'forcediffimfluxunc',
                        'forcediffimsnr', 'forcediffimchisq', 'forcediffimfluxap', 'forcediffimfluxuncap', 'forcediffimsnrap', 'aperturecorr',
                        'dnearestrefsrc', 'nearestrefmag', 'nearestrefmagunc', 'nearestrefchi', 'nearestrefsharp', 'refjdstart', 'refjdend', 'procstatus']
                df = pd.read_csv(file, names=cols, header=None, sep=" ", skiprows=54)
                df = df.set_index(df['index'])  # manually setting indeces
                df = df.drop(columns=['index']) # drop duplicated index 
                df = df[(df['infobitssci'] < 33554432) & (df['scisigpix'] <= 25) & (df['sciinpseeing'] <= 4) & (df['forcediffimflux']!=-99999.0)].reset_index(drop=True) #clean according to docs

                # cut df down to -50 days to +365 days centered on date of first spectra
                # time window included in ztf_fp_info*csv files for each obj type
                obj = ztf_info_df.loc[ztf_info_df['obj_name'].str[3:]==sn_name]
                start_jd = obj['jd_start'].iloc[0]
                end_jd = obj['jd_end'].iloc[0]
                df_cut = df.loc[(df['jd']<end_jd)&(df['jd']>start_jd)] #only selecting points that fall within specified time window
                df_cut = df_cut.reset_index(drop=True)
                df_cut = df_cut.infer_objects() #infering dtype of columns

                mag = df_cut['zpdiff'] - 2.5 * np.log10(df_cut['forcediffimflux'])
                sigma_mag = 1.0857 * df_cut['forcediffimfluxunc']/df_cut['forcediffimflux']
                df_cut['mag'] = mag
                df_cut['e_mag'] = sigma_mag
                df_cut = df_cut.loc[(df_cut['forcediffimflux']>0) & (df_cut['e_mag']<2)].reset_index(drop=True) #only selecting points w/ non-nan mags and errorbars less than 2 mags
                df_cut['mjd'] = df_cut['jd']-2400000.5

                # saving LC in maven-friendly format w/ ZTF name as file name
                # if obj has no ZTF internal name, saved w/ IAU name and will need to look up personally 
                #       and update the metadata
                if save_df:
                        if 'SN '+sn_name in dataset['IAU name'].to_list(): #only converting objs in our "good" sample
                                smol_df = df_cut[['mjd', 'mag', 'e_mag', 'filter']]
                                smol_df = smol_df.rename(columns={"mjd": "time", "e_mag": "magerr", "filter":"band"})
                                smol_df.loc[smol_df['band']=='ZTF_g', 'band'] = 'g'
                                smol_df.loc[smol_df['band']=='ZTF_r', 'band'] = 'R'

                                intnamelist = dataset.loc[dataset['IAU name']=='SN '+sn_name, 'Internal name/s'].to_list()
                                for obj in intnamelist:
                                        if type(obj)==str:
                                                varX = "ZTF"
                                                index = obj.find(varX)
                                                ztf_name = obj[index:index+12]
                                                print(ztf_name, sn_name)
                                                smol_df.to_csv(f'maven_data/lightcurves/{ztf_name}.csv',index=False)
                                        else:
                                                print('WARNING: obj has no internal names, need ZTF name for maven')
                                                smol_df.to_csv(f'maven_data/lightcurves/SN{sn_name}.csv',index=False)                                

                lc_dfs.append(df_cut)

        return sn_names, lc_dfs

In [129]:
# sn_names_ib, lc_dfs_ib = create_ztf_lc_dfs('./ztf_fp_data/ib/','SN Ib')
# sn_names_ic, lc_dfs_ic = create_ztf_lc_dfs('./ztf_fp_data/ic/','SN Ic')
sn_names_iib, lc_dfs_iib = create_ztf_lc_dfs('./ztf_fp_data/iib/','SN IIb', check_names=False, save_df=True)
# sn_names_carich, lc_dfs_carich = create_ztf_lc_dfs('./ztf_fp_data/carich/','SN Ibc Ca-rich', check_names=False)
# sn_names_ibc, lc_dfs_ibc = create_ztf_lc_dfs('./ztf_fp_data/ibc/','SN Ibc', check_names=False)
# sn_names_ibncn, lc_dfs_ibncn = create_ztf_lc_dfs('./ztf_fp_data/ibncn/','SN IbnIcn', check_names=False)
# sn_names_icpec, lc_dfs_icpec = create_ztf_lc_dfs('./ztf_fp_data/icpec/','SN Ic-pec', check_names=False)

/Users/pnr5sh/miniconda3/envs/astro2/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/pnr5sh/miniconda3/envs/astro2/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/pnr5sh/miniconda3/envs/astro2/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/pnr5sh/miniconda3/envs/astro2/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/pnr5sh/miniconda3/envs/astro2/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


ZTF18abojpnr 2018fzn
ZTF18abojpnr 2018fzn
ZTF18acnmifq 2018iug
ZTF18acrcyqw 2018jee
ZTF19aawethv 2019gaf
ZTF19abqmsnk 2019ofk
ZTF19abxtcio 2019pof
ZTF19aceshib 2019sna
ZTF20adadlqm 2020adnx
ZTF20acgiglu 2020urc
ZTF21aaxxmvs 2021kum
ZTF21aayfnjz 2021kww
ZTF21abqvzjy 2021uth
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaawbsc 2023aew
ZTF23aaialcw 2023gwl
ZTF23aazjwtl 2023psq
ZTF23aaaprhz 2023wf
ZTF24abpdzvm 2024zsw


/Users/pnr5sh/miniconda3/envs/astro2/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/pnr5sh/miniconda3/envs/astro2/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/pnr5sh/miniconda3/envs/astro2/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/pnr5sh/miniconda3/envs/astro2/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
